In [1]:
import polars as pl
from sklearn.model_selection import train_test_split
from transformers import features_target_split

# import clean data
df = pl.read_csv("data/clean.csv")
train, test = train_test_split(df, test_size=0.3)

In [2]:
# Build Feature Engineering Pipeline
from transformers import PreprocessingPipeline, ZipCodeTransformer, AustinExperimenting, DropColumnTransformer, PCATransformer

pipeline = PreprocessingPipeline(
    [
        ZipCodeTransformer(),
        PCATransformer(),
        AustinExperimenting(),
        DropColumnTransformer()
    ]
)

In [3]:
# build model
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,

    max_depth=3,
    min_child_weight=10,

    subsample=0.8,
    colsample_bytree=0.6,

    reg_alpha=1,
    reg_lambda=1.5,

    gamma=0.1,

    random_state=42
)

In [4]:
from transformers import test_model

test_model(pipeline,
           model,
           train,
           test,
           "price",
           regression=True,
           show_training=True)


TRAIN
Regression Report
------------------------------
MAE  : 65144.5391
MSE  : 10797632512.0000
MRSE  : 103911.6562
R²   : 0.9219
Average Error: 0.12%

TEST
Regression Report
------------------------------
MAE  : 68886.4531
MSE  : 12854955008.0000
MRSE  : 113379.6953
R²   : 0.8972
Average Error: 0.13%
shape: (13, 2)
┌─────────────────────┬────────────┐
│ Features            ┆ Importance │
│ ---                 ┆ ---        │
│ str                 ┆ f32        │
╞═════════════════════╪════════════╡
│ grade               ┆ 0.271876   │
│ median_price        ┆ 0.14032    │
│ PC 1                ┆ 0.135539   │
│ waterfront          ┆ 0.132517   │
│ view                ┆ 0.089277   │
│ …                   ┆ …          │
│ yr_built            ┆ 0.030967   │
│ yr_since_renovation ┆ 0.014147   │
│ sqft_lot15          ┆ 0.01277    │
│ sqft_lot            ┆ 0.012683   │
│ condition           ┆ 0.009452   │
└─────────────────────┴────────────┘


In [6]:
from transformers import regression_report

# holdout = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv")
holdout = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test.csv")
pipeline.run_train(df)
holdout_preprocessed = pipeline.run_inference(holdout)
preds = model.predict(holdout_preprocessed)
pl.DataFrame(preds, schema=["predictions"]).write_csv("data/holdout_preds.csv")